# Plant Disease Detection - Simple Custom CNN

This first notebook builds a beginner-friendly CNN image classifier for the Kaggle **New Plant Diseases Dataset**.

Scope for the exam project:
- Image preprocessing with `ImageDataGenerator`
- `Conv2D`, `MaxPool2D`, `Flatten`, and `Dense` layers
- ReLU hidden activations and Softmax output activation
- Model training, validation, final accuracy/loss output, optional classification report, and saving the model

No transfer learning or pretrained CNN is used here.

In [ ]:
# Code explanation:
# This block imports the libraries used in the notebook.
# TensorFlow/Keras is used to build and train the CNN.
# scikit-learn is used for the optional classification report.

# Exam note:
# These libraries support the full machine learning workflow: loading data,
# preprocessing images, training a CNN, and evaluating the model.

from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout

from sklearn.metrics import classification_report

print('TensorFlow version:', tf.__version__)

In [ ]:
# Code explanation:
# This block defines the main Colab folders and the Kaggle dataset name.
# In Google Colab, /content is the working folder for the current runtime.
# The dataset will be downloaded from Kaggle and extracted into /content/data/raw.

# Exam note:
# Keeping paths in variables makes the project easier to explain and modify.
# The CNN does not learn from the Kaggle website directly; images must first be downloaded
# and organized into train/valid folders.

PROJECT_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
KAGGLE_DATASET = 'vipoooool/new-plant-diseases-dataset'

KAGGLE_JSON_PATH = PROJECT_ROOT / 'kaggle.json'
KAGGLE_DIR = Path.home() / '.kaggle'
DOWNLOAD_TO = PROJECT_ROOT / 'data' / 'downloads'
EXTRACT_TO = PROJECT_ROOT / 'data' / 'raw'

# Code explanation:
# The models folder is created here because Google Colab starts with an empty /content folder.
# Without this, saving the trained model can fail with FileNotFoundError.

(PROJECT_ROOT / 'models').mkdir(parents=True, exist_ok=True)
DOWNLOAD_TO.mkdir(parents=True, exist_ok=True)
EXTRACT_TO.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Kaggle dataset:', KAGGLE_DATASET)
print('Download folder:', DOWNLOAD_TO)
print('Extract folder:', EXTRACT_TO)

In [ ]:
# Code explanation:
# This block configures Kaggle access in Colab.
# Kaggle may give either:
# 1) a legacy kaggle.json file, or
# 2) a username plus API token that you copy from the Kaggle website.
# This cell supports both methods.

# Exam note:
# Kaggle credentials are not part of the CNN model. They are only used so Colab
# can download the dataset directly from Kaggle. Never share your token publicly
# and never commit it to GitHub.

import json
from getpass import getpass

KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_CREDENTIALS_PATH = KAGGLE_DIR / 'kaggle.json'

if KAGGLE_CREDENTIALS_PATH.exists():
    print('Kaggle credentials already configured at:', KAGGLE_CREDENTIALS_PATH)
else:
    print('Choose one Kaggle setup method:')
    print('1 = Upload legacy kaggle.json file')
    print('2 = Paste Kaggle username and API token manually')

    method = input('Type 1 or 2, then press Enter: ').strip()

    if method == '1':
        try:
            from google.colab import files

            print('Upload kaggle.json now.')
            uploaded = files.upload()
            if 'kaggle.json' not in uploaded:
                raise FileNotFoundError('Please upload a file named kaggle.json')

            shutil.copy(PROJECT_ROOT / 'kaggle.json', KAGGLE_CREDENTIALS_PATH)
        except ModuleNotFoundError:
            raise RuntimeError('File upload helper only works inside Google Colab.')

    elif method == '2':
        kaggle_username = input('Paste your Kaggle username: ').strip()
        kaggle_key = getpass('Paste your Kaggle API token/key. It will be hidden: ').strip()

        credentials = {
            'username': kaggle_username,
            'key': kaggle_key
        }

        with open(KAGGLE_CREDENTIALS_PATH, 'w') as f:
            json.dump(credentials, f)

    else:
        raise ValueError('Please run the cell again and type either 1 or 2.')

    os.chmod(KAGGLE_CREDENTIALS_PATH, 0o600)
    print('Kaggle credentials configured at:', KAGGLE_CREDENTIALS_PATH)


In [ ]:
# Code explanation:
# This block installs the Kaggle command-line tool if needed, downloads the dataset,
# and unzips it into /content/data/raw.
# The download can take a few minutes because the image dataset is large.

# Exam note:
# We download the original dataset in the notebook so the project is reproducible.
# Reproducible means another person can run the same notebook and get the same dataset structure.

expected_train_folder = EXTRACT_TO / 'New Plant Diseases Dataset(Augmented)' / 'New Plant Diseases Dataset(Augmented)' / 'train'

if expected_train_folder.exists():
    print('Dataset already downloaded and extracted.')
else:
    print('Installing Kaggle API package if needed...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

    print('Downloading Kaggle dataset...')
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', KAGGLE_DATASET,
        '-p', str(DOWNLOAD_TO)
    ], check=True)

    zip_files = list(DOWNLOAD_TO.glob('*.zip'))
    if not zip_files:
        raise FileNotFoundError(f'No zip file was downloaded to {DOWNLOAD_TO}')

    dataset_zip = zip_files[0]
    print('Unzipping:', dataset_zip)
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_TO)

    print('Dataset extraction finished.')

In [ ]:
# Code explanation:
# This block defines the train and validation folders used by ImageDataGenerator.
# Each subfolder inside train/ and valid/ is one class, for example Apple___Apple_scab.
# If either folder does not exist, the dataset download or extraction step needs to be checked.

# Exam note:
# The dataset already provides a training set and a validation set.
# Training data is used to learn model weights, while validation data checks performance on unseen images.

DATASET_ROOT = EXTRACT_TO / 'New Plant Diseases Dataset(Augmented)' / 'New Plant Diseases Dataset(Augmented)'
TRAIN_DIR = DATASET_ROOT / 'train'
VALID_DIR = DATASET_ROOT / 'valid'

print('Train folder exists:', TRAIN_DIR.exists())
print('Validation folder exists:', VALID_DIR.exists())
print('Train folder:', TRAIN_DIR)
print('Validation folder:', VALID_DIR)

if not TRAIN_DIR.exists() or not VALID_DIR.exists():
    raise FileNotFoundError('Could not find train/valid folders. Check the Kaggle download and unzip cells above.')

In [ ]:
# Code explanation:
# This block checks the class names and counts how many images each class contains.
# The folder names become the labels that the CNN learns to predict.

# Exam note:
# Before training, we inspect the dataset to understand how many categories/classes exist
# and whether some classes have many more images than others.

image_extensions = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

class_names = sorted([folder.name for folder in TRAIN_DIR.iterdir() if folder.is_dir()])
print('Number of classes:', len(class_names))
print('First 10 class names:', class_names[:10])

train_counts = {}
valid_counts = {}

for class_name in class_names:
    train_counts[class_name] = sum(1 for file in (TRAIN_DIR / class_name).iterdir() if file.suffix in image_extensions)
    valid_counts[class_name] = sum(1 for file in (VALID_DIR / class_name).iterdir() if file.suffix in image_extensions)

print('Total training images:', sum(train_counts.values()))
print('Total validation images:', sum(valid_counts.values()))

for class_name in class_names[:10]:
    print(f'{class_name}: train={train_counts[class_name]}, valid={valid_counts[class_name]}')

In [ ]:
# Code explanation:
# This block sets important training values.
# IMG_SIZE resizes every image to the same width and height.
# BATCH_SIZE controls how many images the model sees before updating its weights.

# Exam note:
# Image resizing is needed because a neural network expects every input to have the same shape.
# 128x128 is a beginner-friendly choice because it trains faster than very large images.

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 10
NUM_CLASSES = len(class_names)

In [ ]:
# Code explanation:
# This block prepares image data using ImageDataGenerator.
# rescale=1./255 converts pixel values from the range 0-255 to 0-1.
# Small augmentation changes are applied to training images only.

# Exam note:
# Scaling pixels to 0-1 helps the model train more smoothly because the input numbers are smaller.
# Augmentation can reduce overfitting by showing slightly changed versions of images during training.
# Overfitting means the model memorizes training images but performs worse on new validation images.

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# Code explanation:
# This block builds a simple custom CNN.
# Conv2D layers learn visual patterns such as edges, spots, colors, and leaf texture.
# MaxPool2D reduces image size while keeping the strongest features.
# Flatten converts 2D feature maps into a 1D vector before Dense layers make the final prediction.

# Exam note:
# We start with 32 filters because early CNN layers learn simple patterns.
# Deeper layers use more filters, like 64 and 128, because they can learn more complex disease patterns.
# ReLU is used because it is simple, fast, and helps the network learn non-linear patterns.
# Softmax is used instead of Sigmoid because this is multi-class classification:
# each image should belong to one class out of many possible plant disease classes.

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    MaxPool2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPool2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPool2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

# Code explanation:
# categorical_crossentropy is used because labels are one-hot encoded by class_mode='categorical'.
# If we used integer labels instead, sparse_categorical_crossentropy would be the matching loss.

# Exam note:
# The loss function measures how wrong the model is during training.
# The optimizer uses that loss to update the CNN weights.

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Code explanation:
# This block trains the CNN using the training generator and checks performance on validation data.
# The history object stores accuracy and loss for each epoch.

# Exam note:
# An epoch means one full pass through the training dataset.
# During training, we compare training accuracy/loss with validation accuracy/loss to check learning quality.

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=valid_generator
)

In [ ]:
# Code explanation:
# This block evaluates the trained model on the validation data.

# Exam note:
# Evaluation tells us how well the model performs on images it did not train on.
# Validation accuracy is more important than training accuracy because it better represents new data.

valid_loss, valid_accuracy = model.evaluate(valid_generator)
print(f'Validation loss: {valid_loss:.4f}')
print(f'Validation accuracy: {valid_accuracy:.4f}')

In [ ]:
# Code explanation:
# This block creates predictions for the validation set and prints a classification report.
# The report shows precision, recall, and F1-score for each class.

# Exam note:
# Accuracy gives one overall score, but the classification report shows whether some diseases
# are harder for the model to recognize than others.

valid_generator.reset()
predicted_probabilities = model.predict(valid_generator)
predicted_classes = np.argmax(predicted_probabilities, axis=1)
true_classes = valid_generator.classes

index_to_class = {value: key for key, value in valid_generator.class_indices.items()}
ordered_class_names = [index_to_class[i] for i in range(NUM_CLASSES)]

print(classification_report(true_classes, predicted_classes, target_names=ordered_class_names))

In [ ]:
# Code explanation:
# This block saves the trained Keras model to the models folder.
# The saved model can later be loaded again without retraining.

# Exam note:
# Saving the model is important because training can take a long time.
# After saving, the trained model can be reused for evaluation or a future app.

(PROJECT_ROOT / 'models').mkdir(parents=True, exist_ok=True)

MODEL_PATH = PROJECT_ROOT / 'models' / 'simple_custom_cnn_plant_disease.keras'
model.save(MODEL_PATH)

print('Model saved to:', MODEL_PATH)